In [ ]:
!pip install -U pageindex openai python-dotenv

In [ ]:
import os, json, time
from dotenv import load_dotenv

load_dotenv()

# Load from environment — NO hardcoded APIs
PAGEINDEX_API_KEY = os.getenv("PAGEINDEX_API_KEY", "")
OPENAI_API_KEY    = os.getenv("OPENAI_API_KEY")

print("PageIndex key loaded:", "v" if PAGEINDEX_API_KEY else "Missing! Set in .env")
print("OpenAI key loaded:   ", "v" if OPENAI_API_KEY    else "Missing! Set in .env")

In [ ]:
from pageindex import PageIndexClient
from openai import OpenAI

pi_client     = PageIndexClient(api_key=PAGEINDEX_API_KEY)
openai_client = OpenAI(api_key=OPENAI_API_KEY)

print("PageIndex client ready")
print("OpenAI client ready")

In [ ]:
PDF_PATH = "./sample_document.pdf"   # ← change this to your PDF

print(f"Uploading: {PDF_PATH}")
if PAGEINDEX_API_KEY:
    result = pi_client.submit_document(PDF_PATH)
    doc_id = result["doc_id"]
    print(f"Uploaded! Document ID: {doc_id}")
else:
    print("Skip upload — PAGEINDEX_API_KEY missing")

In [ ]:
print("Building tree index...")
print("   (Runs once per document — index cached for reuse)")

if PAGEINDEX_API_KEY and 'doc_id' in locals():
    while True:
        try:
            status_result = pi_client.get_document(doc_id)
            status = status_result.get("status")
            print(f"   Status: {status}")
            if status == "completed":
                print("\nTree index ready!")
                break
            elif status == "failed":
                print("\nProcessing failed. Check PDF format.")
                break
            time.sleep(5)
        except Exception as e:
            print(f"Status check error: {e}")
            time.sleep(5)
else:
    print("Skip — missing API key or doc_id")

In [ ]:
if PAGEINDEX_API_KEY and 'doc_id' in locals():
    try:
        tree_result = pi_client.get_tree(doc_id, node_summary=True)
        pageindex_tree = tree_result.get("result", [])
        print(f"📊 Top-level sections: {len(pageindex_tree)}")
        print("\nRaw tree (first node):")
        print(json.dumps(pageindex_tree[0] if pageindex_tree else {}, indent=2))
    except Exception as e:
        print(f"Tree fetch error: {e}")
else:
    print("Skip — set PAGEINDEX_API_KEY and upload first")

In [ ]:
def print_tree(nodes, indent=0):
    for node in nodes:
        prefix = "  " * indent + ("└─ " if indent > 0 else "")
        page = node.get("page_index", "?")
        print(f"{prefix}[{node['node_id']}] {node['title']}  (p.{page})")
        if node.get("nodes"):
            print_tree(node["nodes"], indent + 1)

if PAGEINDEX_API_KEY and 'pageindex_tree' in locals():
    print("Full Document Structure:\n")
    print_tree(pageindex_tree)
else:
    print("Skip — tree not built")

In [ ]:
def count_nodes(nodes):
    total = len(nodes)
    for n in nodes:
        if n.get("nodes"):
            total += count_nodes(n["nodes"])
    return total

if PAGEINDEX_API_KEY and 'pageindex_tree' in locals():
    total = count_nodes(pageindex_tree)
    print(f"Total nodes in tree: {total}")
    print("   Each node = one retrievable section of the document")
else:
    print("Skip — tree not built")

In [ ]:
def llm_tree_search(query: str, tree: list, model: str = "gpt-4o") -> dict:
    def compress(nodes):
        out = []
        for n in nodes:
            entry = {
                "node_id": n["node_id"],
                "title":   n["title"],
                "page":    n.get("page_index", "?"),
                "summary": n.get("text", "")[:150]
            }
            if n.get("nodes"):
                entry["children"] = compress(n["nodes"])
            out.append(entry)
        return out
    
    compressed_tree = compress(tree)
    prompt = f"""You are given a query and a document tree (Table of Contents).
Identify which node IDs contain the answer.
Reply ONLY in JSON: {{\"thinking\": "...", \"node_list\": ["id1"]}}
Query: {query}

Document Tree:\n{json.dumps(compressed_tree, indent=2)}"""
    response = openai_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"}
    )
    return json.loads(response.choices[0].message.content)

if PAGEINDEX_API_KEY and OPENAI_API_KEY and 'pageindex_tree' in locals():
    query = "What is the syllabus covered in Modern LLM finetuning?"
    print(f"Query: {query}\n")
    result = llm_tree_search(query, pageindex_tree)
    print("LLM Reasoning:")
    print(result.get("thinking", "N/A"))
    print()
    print("Selected Node IDs:", result.get("node_list", []))
else:
    print("Skip — set both API keys and build tree first")

In [ ]:
def find_nodes_by_ids(tree: list, target_ids: list) -> list:
    found = []
    for node in tree:
        if node["node_id"] in target_ids:
            found.append(node)
        if node.get("nodes"):
            found.extend(find_nodes_by_ids(node["nodes"], target_ids))
    return found

def generate_answer(query: str, nodes: list, model: str = "gpt-4o") -> str:
    if not nodes:
        return "No relevant sections found in the document."
    context_parts = []
    for node in nodes:
        context_parts.append(
            f"[Section: '{node['title']}' | Page {node.get('page_index', '?')}]\n"
            f"{node.get('text', 'Content not available.')}"
        )
    context = "\n\n---\n\n".join(context_parts)
    prompt = f"""You are an expert document analyst.
Answer using ONLY context. Cite section titles and page numbers.
Question: {query}\n\nContext:\n{context}\n\nAnswer:"""
    response = openai_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

if PAGEINDEX_API_KEY and OPENAI_API_KEY and 'pageindex_tree' in locals():
    selected_ids = result.get("node_list", []) if 'result' in locals() else []
    nodes = find_nodes_by_ids(pageindex_tree, selected_ids)
    answer = generate_answer(query, nodes)
    print("\n=== ANSURED ANSWER ===")
    print(answer)
else:
    print("Skip — complete upload + tree + search first")